In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Pasy Transpilatora zasilane przez AI

Pasy Transpilatora zasilane przez AI to pasy, które działają jako zamiennik tradycyjnych pasów Qiskit dla niektórych zadań transpilacji. Często dają lepsze wyniki niż istniejące algorytmy heurystyczne (takie jak mniejsza głębokość obwodu i liczba bramek CNOT), a jednocześnie są znacznie szybsze niż algorytmy optymalizacji, takie jak rozwiązywacze spełniania formuł boolowskich. Pasy Transpilatora AI działają w środowisku lokalnym.

> **Note:** Pasy Transpilatora zasilane przez AI są w wersji beta i mogą ulec zmianom.
>     Jeśli masz uwagi lub chcesz skontaktować się z zespołem deweloperskim, skorzystaj z tego [kanału Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU).

Aktualnie dostępne są następujące pasy:

**Pasy trasowania**
 - `AIRouting`: Wybór układu i trasowanie Circuit

**Pasy syntezy Circuit**
 - `AICliffordSynthesis`: Synteza Circuit Clifforda
 - `AILinearFunctionSynthesis`: Synteza Circuit funkcji liniowych
 - `AIPermutationSynthesis`: Synteza Circuit permutacji

Aby korzystać z pasów Transpilatora AI, najpierw zainstaluj pakiet `qiskit-ibm-transpiler`. Odwiedź [dokumentację API qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler), aby uzyskać więcej informacji o dostępnych opcjach.

## Uruchamianie pasów Transpilatora AI lokalnie lub w chmurze
Najpierw zainstaluj `qiskit-ibm-transpiler` z dodatkowymi zależnościami w następujący sposób:

In [1]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService
import logging

backend = QiskitRuntimeService().backend("ibm_fez")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
            local_mode=True,
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)
logging.getLogger(
    "qiskit_ibm_transpiler.wrappers.ai_local_synthesis"
).setLevel(logging.WARNING)
transpiled_circuit = ai_passmanager.run(circuit)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Po zainstalowaniu dodatkowych zależności domyślnym trybem uruchamiania pasów Transpilatora zasilanych przez AI jest korzystanie z lokalnego komputera.

## Pas trasowania AI
Pas `AIRouting` pełni zarówno rolę etapu układu, jak i etapu trasowania. Można go użyć w ramach `PassManager` w następujący sposób:

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit.circuit.library import efficient_su2

ibm_kingston = QiskitRuntimeService().backend("ibm_kingston")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_kingston,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_kingston, local_mode=True
        ),  # Re-synthesize Linear Function blocks
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Fetching 127 files:   0%|          | 0/127 [00:00<?, ?it/s]

The synthesis respects the coupling map of the device: it can be run safely after other routing passes without disturbing the circuit, so the overall circuit will still follow the device restrictions. By default, the synthesis will replace the original sub-circuit only if the synthesized sub-circuit improves the original (currently only checking CNOT count), but this can be forced to always replace the circuit by setting `replace_only_if_better=False`.

The following synthesis passes are available from `qiskit_ibm_transpiler.ai.synthesis`:

- *AICliffordSynthesis*: Synthesis for [Clifford](/docs/api/qiskit/qiskit.quantum_info.Clifford) circuits (blocks of `H`, `S`, and `CX` gates). Currently up to nine qubit blocks.
- *AILinearFunctionSynthesis*: Synthesis for [Linear Function](/docs/api/qiskit/qiskit.circuit.library.LinearFunction) circuits (blocks of `CX` and `SWAP` gates). Currently up to nine qubit blocks.
- *AIPermutationSynthesis*: Synthesis for [Permutation](/docs/api/qiskit/qiskit.circuit.library.Permutation#permutation) circuits (blocks of `SWAP` gates). Currently available for 65, 33, and 27 qubit blocks.
- *AIPauliNetworkSynthesis*: Synthesis for Pauli Network circuits (blocks of `H`, `S`, `SX`, `CX`, `RX`, `RY` and `RZ` gates). Currently up to six qubit blocks.

We expect to gradually increase the size of the supported blocks.

All passes use a thread pool to send several requests in parallel. By default, the number for max threads is the number of cores plus four (default values for the `ThreadPoolExecutor` Python object). However, you can set your own value with the `max_threads` argument at pass instantiation. For example, the following line instantiates the `AILinearFunctionSynthesis` pass, which allows it to use a maximum of 20 threads.

```python
AILinearFunctionSynthesis(backend=ibm_torino, max_threads=20)  # Re-synthesize Linear Function blocks using 20 threads max
```

You can also set the environment variable `AI_TRANSPILER_MAX_THREADS` to the desired number of maximum threads, and all synthesis passes instantiated after that will use that value.

For the AI synthesis passes to synthesize a sub-circuit, it must lay on a connected subgraph of the coupling map (one way to do this is with a routing pass before collecting the blocks, but this is not the only way to do it). The synthesis passes will automatically check that the specific subgraph is supported, and if not, it will raise a warning and leave the original sub-circuit unchanged.

The following custom collection passes for Cliffords, Linear Functions and Permutations that can be imported from `qiskit_ibm_transpiler.ai.collection` also complement the synthesis passes:

- *CollectCliffords*: Collects Clifford blocks as `Instruction` objects and stores the original sub-circuit to compare against it after synthesis.
- *CollectLinearFunctions*: Collects blocks of `SWAP` and `CX` as `LinearFunction` objects and stores the original sub-circuit to compare against it after synthesis.
- *CollectPermutations*: Collects blocks of `SWAP` circuits as `Permutations`.
- *CollectPauliNetworks*: Collects Pauli Network blocks and stores the original sub-circuit to compare against it after synthesis.

These custom collection passes limit the sizes of the collected sub-circuits so they are supported by the AI-powered synthesis passes. Therefore, it is recommended to use them after the routing passes and before the synthesis passes for a better overall optimization.

## Hybrid heuristic-AI circuit transpilation

The `qiskit-ibm-transpiler` allows you to configure a hybrid pass manager that combines the best of Qiskit's heuristic and the AI-powered transpiler passes. This feature behaves similarly to the Qiskit `generate_pass_manager` method. A typical way to use `generate_ai_pass_manager` is as follows:

In [ ]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_kingston")
kingston_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=kingston_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

Tutaj `backend` określa, dla której mapy sprzężeń przeprowadzać trasowanie, `optimization_level` (1, 2 lub 3) określa nakład obliczeniowy przeznaczony na ten proces (wyższy poziom zazwyczaj daje lepsze wyniki, ale trwa dłużej), a `layout_mode` określa sposób obsługi wyboru układu.
`layout_mode` oferuje następujące opcje:

- `keep`: Respektuje układ ustawiony przez poprzednie pasy Transpilatora (lub używa trywialnego układu, jeśli nie jest ustawiony). Zwykle używany tylko wtedy, gdy Circuit musi być uruchamiany na konkretnych Qubitach urządzenia. Często daje gorsze wyniki, ponieważ ma mniej miejsca na optymalizację.
- `improve`: Używa układu ustawionego przez poprzednie pasy Transpilatora jako punktu startowego. Jest przydatny, gdy masz dobre wstępne przypuszczenie co do układu; na przykład dla Circuit zbudowanych w sposób, który w przybliżeniu odwzorowuje mapę sprzężeń urządzenia. Jest też przydatny, gdy chcesz wypróbować inne specyficzne pasy układu w połączeniu z pasem `AIRouting`.
- `optimize`: Jest to tryb domyślny. Działa najlepiej dla ogólnych Circuit, gdy nie masz dobrych przypuszczeń co do układu. Ten tryb ignoruje poprzednie wybory układu.
- `local_mode`: Ta flaga określa, gdzie uruchamiany jest pas `AIRouting`. Jeśli `False`, `AIRouting` działa zdalnie za pośrednictwem Qiskit Transpiler Service. Jeśli `True`, pakiet próbuje uruchomić pas w lokalnym środowisku, z możliwością awaryjnego przełączenia na tryb chmury, jeśli wymagane zależności nie zostaną znalezione.

## Pasy syntezy Circuit AI
Pasy syntezy Circuit AI umożliwiają optymalizację fragmentów różnych typów Circuit ([Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford), [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction), [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation), Pauli Network) poprzez ich resyntetyzowanie. Typowy sposób użycia pasu syntezy jest następujący: